# Construcción de insumos para el visor territorial

**Señales del entorno: violencia intrafamiliar, consumos SPA y lesiones personales**

Este cuaderno parte de las bases ya limpias por fuente y reconstruye los insumos analíticos usados por el visor. Su función es dejar verificable cómo se construyen la base integrada, los indicadores, las pruebas exploratorias y los archivos de salida.

La regla de integración se mantiene: las fuentes no se suman entre sí. Cada fuente conserva su fenómeno y se compara mediante indicadores territoriales por localidad.

## 1. Alcance

El flujo de este cuaderno es:

1. Leer bases limpias de VIF, lesiones personales y consumo abusivo de SPA.
2. Consolidar indicadores por localidad para 2018-2025.
3. Calcular tasas, persistencia, Índice VIF, índice de contexto y brecha VIF-contexto.
4. Ejecutar validaciones de consistencia.
6. Exportar los archivos requeridos para el repositorio.

El visor HTML se conserva como producto de visualización. Este cuaderno no reescribe.

In [1]:
from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd

try:
    from scipy.stats import spearmanr, kendalltau, pearsonr
    SCIPY_DISPONIBLE = True
except Exception:
    SCIPY_DISPONIBLE = False

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 2. Rutas de trabajo

El cuaderno está preparado para dos formas de uso:

- En repositorio: archivos en `data/clean/`, `outputs_limpieza/` o `data/raw/`.
- En Colab: archivos cargados directamente en `/content` o dentro de `/content/outputs_limpieza`.

Los resultados se escriben en `outputs/`. Si se ejecuta en Colab, quedarán en `/content/outputs`.

In [2]:
def nombre_normalizado(texto):
    texto = str(texto).strip().lower()
    texto = "".join(
        ch for ch in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(ch)
    )
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    return texto.strip("_")


def carpeta_util(carpeta):
    if not carpeta.exists() or not carpeta.is_dir():
        return False
    señales = [
        carpeta / "data",
        carpeta / "outputs_limpieza",
        carpeta / "vif_limpia_localizada_2018_2025.csv",
        carpeta / "spa_abusivo_limpio_localizado_2018_2025.csv",
        carpeta / "README.md",
    ]
    return any(s.exists() for s in señales)


def detectar_base_dir():
    """Detecta la carpeta base sin amarrar el cuaderno a rutas locales."""
    # En Colab debe priorizar /content, porque allí quedan los archivos cargados por el usuario.
    if carpeta_util(Path("/content")):
        return Path("/content")

    # En este entorno de trabajo y en algunas pruebas locales, los archivos quedan en /mnt/data.
    if carpeta_util(Path("/mnt/data")):
        return Path("/mnt/data")

    # En repositorio local se parte de la carpeta actual y sus padres, sin aceptar la raíz del sistema.
    cwd = Path.cwd().resolve()
    for carpeta in [cwd, *cwd.parents]:
        if carpeta == carpeta.parent:  # evita usar / como raíz del proyecto
            continue
        if carpeta_util(carpeta):
            return carpeta

    return cwd


BASE_DIR = detectar_base_dir()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CARPETAS_BUSQUEDA = [
    BASE_DIR,
    BASE_DIR / "data",
    BASE_DIR / "data" / "clean",
    BASE_DIR / "data" / "processed",
    BASE_DIR / "outputs_limpieza",
    BASE_DIR / "visualizacion",
]
CARPETAS_BUSQUEDA = [] + [c for c in CARPETAS_BUSQUEDA if c.exists()]

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Carpetas revisadas:")
for carpeta in CARPETAS_BUSQUEDA:
    print("-", carpeta)

BASE_DIR: /content
OUTPUT_DIR: /content/outputs
Carpetas revisadas:
- /content


## 3. Localización de archivos

Se buscan los archivos limpios por nombre normalizado. Esto evita errores por mayúsculas, minúsculas o variaciones menores en el nombre del archivo.

In [3]:
def buscar_archivo(opciones, requerido=True, permitir_parcial=True):
    """Busca un archivo respetando primero nombres exactos normalizados y el orden de opciones."""
    opciones_norm = [(opcion, nombre_normalizado(opcion), Path(opcion).suffix.lower()) for opcion in opciones]

    # Búsqueda exacta, respetando el orden de opciones.
    for opcion_original, opcion_norm, extension in opciones_norm:
        for carpeta in CARPETAS_BUSQUEDA:
            for archivo in carpeta.glob("*"):
                if not archivo.is_file():
                    continue
                if extension and archivo.suffix.lower() != extension:
                    continue
                if nombre_normalizado(archivo.name) == opcion_norm:
                    return archivo

    # Búsqueda parcial solo si se permite. Útil para pequeñas variaciones en nombres de CSV.
    if permitir_parcial:
        for opcion_original, opcion_norm, extension in opciones_norm:
            clave = opcion_norm.replace("_csv", "").replace("_xlsx", "").replace("_html", "").strip("_")
            for carpeta in CARPETAS_BUSQUEDA:
                for archivo in carpeta.glob("*"):
                    if not archivo.is_file():
                        continue
                    if extension and archivo.suffix.lower() != extension:
                        continue
                    if clave and clave in nombre_normalizado(archivo.name):
                        return archivo

    if requerido:
        raise FileNotFoundError(
            "No se encontró ninguno de estos archivos: " + ", ".join(opciones) +
            "\nRevise que esté cargado en data/clean, outputs_limpieza o en la raíz de Colab."
        )
    return None


PATHS = {
    "vif": buscar_archivo(["vif_limpia_localizada_2018_2025.csv"]),
    "lesiones": buscar_archivo(["lesiones_limpia_localizada_2018_2025.csv"]),
    "spa_abusivo": buscar_archivo(["spa_abusivo_limpio_localizado_2018_2025.csv"]),
    "prevalencia_spa": buscar_archivo(["prevalencia_spa_exploratoria_2022.csv"], requerido=False),
    "rnmc_revision": buscar_archivo(["rnmc_revision_metodologica.csv"], requerido=False),
    "resumen_limpieza": buscar_archivo(["resumen_limpieza_fuentes.csv"], requerido=False),
}

HTML_CANDIDATOS = [
    "reporte_graficos_propuesta_final_vif_2018_2025_RECOMENDACIONES_V3.html",
    "reporte_graficos_propuesta_final_vif_2018_2025_RECOMENDACIONES_V2.html",
    "index.html",
    "index (1028).html",
]
HTML_PATH = buscar_archivo(HTML_CANDIDATOS, requerido=False, permitir_parcial=False)

for nombre, ruta in PATHS.items():
    print(f"{nombre:18s} -> {ruta}")
print(f"{'html_visor':18s} -> {HTML_PATH}")

vif                -> /content/vif_limpia_localizada_2018_2025.csv
lesiones           -> /content/lesiones_limpia_localizada_2018_2025.csv
spa_abusivo        -> /content/spa_abusivo_limpio_localizado_2018_2025.csv
prevalencia_spa    -> /content/prevalencia_spa_exploratoria_2022.csv
rnmc_revision      -> /content/rnmc_revision_metodologica.csv
resumen_limpieza   -> /content/resumen_limpieza_fuentes.csv
html_visor         -> None


## 4. Carga de bases limpias

Estas bases ya vienen localizadas y filtradas desde el proceso de limpieza. De todas formas se aplican filtros de control para mantener la ventana 2018-2025 y excluir registros no válidos.

In [20]:
vif = pd.read_csv(PATHS["vif"])
lesiones = pd.read_csv(PATHS["lesiones"])
spa_abusivo = pd.read_csv(PATHS["spa_abusivo"])

prevalencia_spa = pd.read_csv(PATHS["prevalencia_spa"]) if PATHS["prevalencia_spa"] else None
rnmc_revision = pd.read_csv(PATHS["rnmc_revision"]) if PATHS["rnmc_revision"] else None
resumen_limpieza = pd.read_csv(PATHS["resumen_limpieza"]) if PATHS["resumen_limpieza"] else None

print("VIF:", vif.shape)
print("Lesiones:", lesiones.shape)
print("SPA abusivo:", spa_abusivo.shape)
if prevalencia_spa is not None:
    print("Prevalencia SPA:", prevalencia_spa.shape)
if rnmc_revision is not None:
    print("RNMC revisión:", rnmc_revision.shape)

VIF: (52757, 14)
Lesiones: (51908, 14)
SPA abusivo: (67230, 13)
Prevalencia SPA: (20, 11)
RNMC revisión: (3, 2)


## 5. Controles mínimos de estructura

Antes de calcular indicadores se revisa que las columnas necesarias estén presentes. Esto evita producir resultados incompletos sin advertencia.

In [21]:
def validar_columnas(df, columnas, nombre):
    faltantes = [c for c in columnas if c not in df.columns]
    if faltantes:
        raise ValueError(f"{nombre}: faltan columnas requeridas: {faltantes}")
    print(f"{nombre}: columnas requeridas completas")

validar_columnas(vif, ["ANIO", "loc_norm", "localidad_valida", "CANTIDAD", "SEXO", "RANGO_VITAL"], "VIF")
validar_columnas(lesiones, ["ANIO", "loc_norm", "localidad_valida", "CANTIDAD"], "Lesiones")
validar_columnas(spa_abusivo, ["ANO", "loc_norm", "localidad_valida", "CASOS"], "SPA abusivo")

if prevalencia_spa is not None:
    validar_columnas(prevalencia_spa, ["loc_norm", "Consumo actual"], "Prevalencia SPA")

VIF: columnas requeridas completas
Lesiones: columnas requeridas completas
SPA abusivo: columnas requeridas completas
Prevalencia SPA: columnas requeridas completas


## 6. Población de referencia

Se usa una tabla fija de población de referencia por localidad. Esta tabla actúa como denominador para las tasas promedio anuales por 100.000 habitantes.

In [22]:
poblacion_referencia = pd.DataFrame([
    ("USAQUEN", "Usaquén", 501999),
    ("CHAPINERO", "Chapinero", 139701),
    ("SANTA FE", "Santa Fe", 110048),
    ("SAN CRISTOBAL", "San Cristóbal", 404697),
    ("USME", "Usme", 457302),
    ("TUNJUELITO", "Tunjuelito", 199430),
    ("BOSA", "Bosa", 673077),
    ("KENNEDY", "Kennedy", 1088443),
    ("FONTIBON", "Fontibón", 394648),
    ("ENGATIVA", "Engativá", 887080),
    ("SUBA", "Suba", 1218513),
    ("BARRIOS UNIDOS", "Barrios Unidos", 243465),
    ("TEUSAQUILLO", "Teusaquillo", 153025),
    ("LOS MARTIRES", "Los Mártires", 99119),
    ("ANTONIO NARINO", "Antonio Nariño", 109176),
    ("PUENTE ARANDA", "Puente Aranda", 258287),
    ("LA CANDELARIA", "La Candelaria", 24088),
    ("RAFAEL URIBE URIBE", "Rafael Uribe Uribe", 374246),
    ("CIUDAD BOLIVAR", "Ciudad Bolívar", 707569),
    ("SUMAPAZ", "Sumapaz", 6531),
], columns=["loc_norm", "localidad", "poblacion_ref"])

poblacion_referencia

,loc_norm,localidad,poblacion_ref
0,USAQUEN,Usaquén,501999
1,CHAPINERO,Chapinero,139701
2,SANTA FE,Santa Fe,110048
3,SAN CRISTOBAL,San Cristóbal,404697
4,USME,Usme,457302
5,TUNJUELITO,Tunjuelito,199430
6,BOSA,Bosa,673077
7,KENNEDY,Kennedy,1088443
8,FONTIBON,Fontibón,394648
9,ENGATIVA,Engativá,887080


## 7. Filtro temporal y registros localizados

Aunque los archivos ya están limpios, se repite el filtro para que el cuaderno sea estable si se actualizan las bases.

In [7]:
vif = vif[(vif["ANIO"].between(2018, 2025)) & (vif["localidad_valida"] == True)].copy()
lesiones = lesiones[(lesiones["ANIO"].between(2018, 2025)) & (lesiones["localidad_valida"] == True)].copy()
spa_abusivo = spa_abusivo[(spa_abusivo["ANO"].between(2018, 2025)) & (spa_abusivo["localidad_valida"] == True)].copy()

print("Registros después de control 2018-2025 y localidad válida")
print("VIF:", len(vif), "| casos:", int(vif["CANTIDAD"].sum()))
print("Lesiones:", len(lesiones), "| casos:", int(lesiones["CANTIDAD"].sum()))
print("SPA abusivo:", len(spa_abusivo), "| casos:", int(spa_abusivo["CASOS"].sum()))

Registros después de control 2018-2025 y localidad válida
VIF: 52757 | casos: 283773
Lesiones: 51908 | casos: 162475
SPA abusivo: 67230 | casos: 74238


## 8. Agregación por localidad

Cada fuente se agrega de manera independiente. No se suman VIF, lesiones y SPA en un mismo conteo.

In [8]:
vif_casos = (
    vif.groupby("loc_norm", as_index=False)["CANTIDAD"]
    .sum()
    .rename(columns={"CANTIDAD": "vif_casos_2018_2025"})
)

lesiones_casos = (
    lesiones.groupby("loc_norm", as_index=False)["CANTIDAD"]
    .sum()
    .rename(columns={"CANTIDAD": "lesiones_casos_2018_2025"})
)

spa_casos = (
    spa_abusivo.groupby("loc_norm", as_index=False)["CASOS"]
    .sum()
    .rename(columns={"CASOS": "spa_abusivo_casos_2018_2025"})
)

vif_casos.sort_values("vif_casos_2018_2025", ascending=False).head(10)

,loc_norm,vif_casos_2018_2025
7,KENNEDY,34615
14,SUBA,33859
4,CIUDAD BOLIVAR,32707
2,BOSA,31314
5,ENGATIVA,26311
12,SAN CRISTOBAL,17723
19,USME,15558
11,RAFAEL URIBE URIBE,12593
18,USAQUEN,12562
6,FONTIBON,12180


## 9. Persistencia anual de VIF

La persistencia mide en cuántos años la tasa anual de una localidad estuvo por encima de la mediana anual de las localidades. No mide tendencia ni causalidad; ayuda a distinguir episodios puntuales de señales repetidas en el tiempo.

In [9]:
vif_anual = (
    vif.groupby(["loc_norm", "ANIO"], as_index=False)["CANTIDAD"]
    .sum()
    .merge(poblacion_referencia[["loc_norm", "poblacion_ref"]], on="loc_norm", how="left")
)

vif_anual["tasa_anual_100k"] = vif_anual["CANTIDAD"] / vif_anual["poblacion_ref"] * 100000

mediana_anual = (
    vif_anual.groupby("ANIO", as_index=False)["tasa_anual_100k"]
    .median()
    .rename(columns={"tasa_anual_100k": "mediana_anual_100k"})
)

vif_anual = vif_anual.merge(mediana_anual, on="ANIO", how="left")
vif_anual["sobre_mediana"] = vif_anual["tasa_anual_100k"] > vif_anual["mediana_anual_100k"]

persistencia_vif = (
    vif_anual.groupby("loc_norm", as_index=False)["sobre_mediana"]
    .sum()
    .rename(columns={"sobre_mediana": "vif_persistencia_anios"})
)

persistencia_vif.sort_values("vif_persistencia_anios", ascending=False).head(10)

,loc_norm,vif_persistencia_anios
12,SAN CRISTOBAL,8
9,LOS MARTIRES,8
8,LA CANDELARIA,8
4,CIUDAD BOLIVAR,8
13,SANTA FE,8
2,BOSA,7
19,USME,6
3,CHAPINERO,4
17,TUNJUELITO,4
10,PUENTE ARANDA,3


## 10. Participación de mujeres, niñas, niños y adolescentes

Se calcula la participación de registros de VIF asociados a mujeres o a rangos vitales de infancia y adolescencia. Es un indicador de enfoque poblacional dentro del fenómeno VIF.

In [10]:
vif["flag_mujeres_nna"] = (
    vif["SEXO"].astype(str).str.upper().eq("FEMENINO") |
    vif["RANGO_VITAL"].astype(str).str.upper().isin(["INFANCIA", "ADOLESCENCIA"])
)

num_mujeres_nna = vif.loc[vif["flag_mujeres_nna"]].groupby("loc_norm")["CANTIDAD"].sum()
den_vif = vif.groupby("loc_norm")["CANTIDAD"].sum()

pct_mujeres_nna = (
    (num_mujeres_nna / den_vif * 100)
    .round(2)
    .rename("pct_mujeres_nna_vif")
    .reset_index()
)

pct_mujeres_nna.sort_values("pct_mujeres_nna_vif", ascending=False).head(10)

,loc_norm,pct_mujeres_nna_vif
15,SUMAPAZ,82.46
19,USME,80.63
4,CIUDAD BOLIVAR,80.45
2,BOSA,79.53
7,KENNEDY,79.46
12,SAN CRISTOBAL,79.32
11,RAFAEL URIBE URIBE,78.67
6,FONTIBON,77.82
14,SUBA,77.72
17,TUNJUELITO,77.71


## 11. Base integrada por localidad

La integración se hace por localidad y por indicadores. En este punto cada fuente conserva su campo de casos y su tasa.

In [11]:
base_integrada = (
    poblacion_referencia
    .merge(vif_casos, on="loc_norm", how="left")
    .merge(persistencia_vif, on="loc_norm", how="left")
    .merge(pct_mujeres_nna, on="loc_norm", how="left")
    .merge(lesiones_casos, on="loc_norm", how="left")
    .merge(spa_casos, on="loc_norm", how="left")
)

for col in ["vif_casos_2018_2025", "vif_persistencia_anios", "lesiones_casos_2018_2025", "spa_abusivo_casos_2018_2025"]:
    base_integrada[col] = base_integrada[col].fillna(0).astype(int)

base_integrada["vif_tasa_prom_anual_100k"] = (
    base_integrada["vif_casos_2018_2025"] / base_integrada["poblacion_ref"] * 100000 / 8
).round(2)

base_integrada["lesiones_tasa_prom_anual_100k"] = (
    base_integrada["lesiones_casos_2018_2025"] / base_integrada["poblacion_ref"] * 100000 / 8
).round(2)

base_integrada["spa_abusivo_tasa_prom_anual_100k"] = (
    base_integrada["spa_abusivo_casos_2018_2025"] / base_integrada["poblacion_ref"] * 100000 / 8
).round(2)

if prevalencia_spa is not None:
    prev_localidad = (
        prevalencia_spa.groupby("loc_norm", as_index=False)["Consumo actual"]
        .mean()
        .rename(columns={"Consumo actual": "consumo_actual_spa_2022"})
    )
    base_integrada = base_integrada.merge(prev_localidad, on="loc_norm", how="left")

base_integrada.head()

,loc_norm,localidad,poblacion_ref,vif_casos_2018_2025,vif_persistencia_anios,pct_mujeres_nna_vif,lesiones_casos_2018_2025,spa_abusivo_casos_2018_2025,vif_tasa_prom_anual_100k,lesiones_tasa_prom_anual_100k,spa_abusivo_tasa_prom_anual_100k,consumo_actual_spa_2022
0,USAQUEN,Usaquén,501999,12562,0,76.17,6999,2979,312.80,174.28,74.18,9.3
1,CHAPINERO,Chapinero,139701,5119,4,76.38,4435,2185,458.03,396.83,195.51,9.7
2,SANTA FE,Santa Fe,110048,7813,8,74.93,6441,4877,887.45,731.61,553.96,5.1
3,SAN CRISTOBAL,San Cristóbal,404697,17723,8,79.32,9979,2993,547.42,308.22,92.45,4.1
4,USME,Usme,457302,15558,6,80.63,9623,3408,425.27,263.04,93.16,3.6


## 12. Índices de lectura territorial

El Índice VIF combina cuatro componentes normalizados:

- Tasa VIF: 40%.
- Persistencia anual: 30%.
- Participación de mujeres y NNA: 20%.
- Casos registrados: 10%.

El índice de contexto combina lesiones personales y consumo abusivo de SPA en partes iguales. Se usa como señal territorial complementaria, no como causa de VIF.

In [12]:
def minmax_100(serie):
    serie = pd.Series(serie, dtype=float)
    minimo = serie.min()
    maximo = serie.max()
    if maximo == minimo:
        return pd.Series(0, index=serie.index, dtype=float)
    return (serie - minimo) / (maximo - minimo) * 100

base_integrada["indice_vif_familiar"] = (
    0.40 * minmax_100(base_integrada["vif_tasa_prom_anual_100k"]) +
    0.30 * minmax_100(base_integrada["vif_persistencia_anios"]) +
    0.20 * minmax_100(base_integrada["pct_mujeres_nna_vif"]) +
    0.10 * minmax_100(base_integrada["vif_casos_2018_2025"])
).round(2)

base_integrada["indice_contexto"] = (
    0.50 * minmax_100(base_integrada["lesiones_tasa_prom_anual_100k"]) +
    0.50 * minmax_100(base_integrada["spa_abusivo_tasa_prom_anual_100k"])
).round(2)

base_integrada["brecha_vif_contexto"] = (
    base_integrada["indice_vif_familiar"] - base_integrada["indice_contexto"]
).round(2)

base_integrada[["localidad", "indice_vif_familiar", "indice_contexto", "brecha_vif_contexto"]].sort_values("indice_vif_familiar", ascending=False).head(10)

,localidad,indice_vif_familiar,indice_contexto,brecha_vif_contexto
13,Los Mártires,76.18,94.00,-17.82
18,Ciudad Bolívar,74.12,17.31,56.81
16,La Candelaria,70.28,97.33,-27.05
6,Bosa,68.27,17.30,50.97
2,Santa Fe,68.22,69.66,-1.44
3,San Cristóbal,66.31,18.96,47.35
4,Usme,55.93,16.37,39.56
7,Kennedy,46.75,14.46,32.29
5,Tunjuelito,40.66,21.90,18.76
17,Rafael Uribe Uribe,39.72,19.34,20.38


## 13. Rankings y lectura operativa

Los rankings ordenan la lectura para el visor. No reemplazan la discusión territorial ni la validación institucional.

In [13]:
base_integrada["rank_vif_carga"] = base_integrada["vif_casos_2018_2025"].rank(ascending=False, method="min").astype(int)
base_integrada["rank_vif_tasa"] = base_integrada["vif_tasa_prom_anual_100k"].rank(ascending=False, method="min").astype(int)
base_integrada["rank_vif_familiar"] = base_integrada["indice_vif_familiar"].rank(ascending=False, method="min").astype(int)
base_integrada["cambio_carga_vs_tasa"] = base_integrada["rank_vif_carga"] - base_integrada["rank_vif_tasa"]


def nivel_indice(valor):
    if valor >= 66:
        return "alto"
    if valor >= 33:
        return "medio"
    return "bajo"

base_integrada["nivel_vif_familiar"] = base_integrada["indice_vif_familiar"].apply(nivel_indice)
base_integrada["nivel_contexto"] = base_integrada["indice_contexto"].apply(nivel_indice)


def tipo_decision(row):
    if row["nivel_vif_familiar"] == "alto" and row["nivel_contexto"] == "alto":
        return "VIF alta con entorno complejo"
    if row["nivel_vif_familiar"] == "alto":
        return "VIF alta con contexto moderado"
    if row["nivel_vif_familiar"] == "medio":
        return "Seguimiento focalizado"
    return "Monitoreo general"


def alerta_lectura(row):
    if row["poblacion_ref"] < 50000:
        return "Alta: población pequeña. Interpretar tasas con cautela."
    if row["localidad"] in ["Los Mártires", "Santa Fe", "Chapinero"]:
        return "Media: posible efecto de población flotante o centralidad urbana."
    return "Baja: lectura territorial más estable."


def accion_sugerida(row):
    if row["nivel_vif_familiar"] == "alto" and row["nivel_contexto"] == "alto":
        return "Conviene revisar la oferta de atención familiar junto con salud mental, convivencia y presencia institucional en el territorio."
    if row["nivel_vif_familiar"] == "alto":
        return "La prioridad debería estar en rutas de atención, prevención en hogares, enfoque de género y protección de niñas, niños y adolescentes."
    if row["brecha_vif_contexto"] > 8:
        return "La VIF pesa más que las señales de contexto. Vale la pena mirar con más detalle la dinámica familiar y las rutas de atención."
    return "Mantener seguimiento y contrastar con nuevos cortes de información y conocimiento territorial."

base_integrada["tipo_decision"] = base_integrada.apply(tipo_decision, axis=1)
base_integrada["alerta_lectura"] = base_integrada.apply(alerta_lectura, axis=1)
base_integrada["accion_sugerida"] = base_integrada.apply(accion_sugerida, axis=1)

## 14. Texto breve para la ficha territorial

El texto se genera con reglas simples para mantener consistencia entre localidades. No introduce nuevas conclusiones; resume los indicadores calculados.

In [14]:
def formato_miles(valor):
    return f"{int(valor):,}".replace(",", ".")


def etiqueta_carga(rank):
    if rank <= 5:
        return "alta"
    if rank <= 15:
        return "media"
    return "baja"


def construir_lectura(row):
    texto = (
        f"Registra {formato_miles(row['vif_casos_2018_2025'])} casos entre 2018 y 2025, "
        f"con una carga {etiqueta_carga(row['rank_vif_carga'])} frente al resto de localidades. "
        f"La tasa promedio es de {row['vif_tasa_prom_anual_100k']:.0f} por 100.000 habitantes. "
        f"El índice VIF es {row['nivel_vif_familiar']} y el contexto territorial es {row['nivel_contexto']}."
    )
    if row["alerta_lectura"].startswith("Alta"):
        texto += " La tasa debe leerse con cautela porque la población es pequeña."
    elif row["alerta_lectura"].startswith("Media"):
        texto += " Conviene revisar población flotante y dinámica urbana antes de sacar conclusiones."
    return texto

base_integrada["lectura_breve"] = base_integrada.apply(construir_lectura, axis=1)
base_integrada["lectura_breve_final"] = base_integrada["lectura_breve"]

columnas_finales = [
    "localidad", "loc_norm", "poblacion_ref",
    "vif_casos_2018_2025", "vif_tasa_prom_anual_100k", "vif_persistencia_anios", "pct_mujeres_nna_vif",
    "lesiones_casos_2018_2025", "lesiones_tasa_prom_anual_100k",
    "spa_abusivo_casos_2018_2025", "spa_abusivo_tasa_prom_anual_100k",
    "consumo_actual_spa_2022",
    "indice_vif_familiar", "indice_contexto", "brecha_vif_contexto",
    "rank_vif_carga", "rank_vif_tasa", "rank_vif_familiar", "cambio_carga_vs_tasa",
    "nivel_vif_familiar", "nivel_contexto", "tipo_decision",
    "alerta_lectura", "accion_sugerida", "lectura_breve", "lectura_breve_final",
]
columnas_finales = [c for c in columnas_finales if c in base_integrada.columns]
base_integrada = base_integrada[columnas_finales].sort_values("rank_vif_familiar").reset_index(drop=True)

base_integrada.head(10)

,localidad,loc_norm,poblacion_ref,vif_casos_2018_2025,vif_tasa_prom_anual_100k,vif_persistencia_anios,pct_mujeres_nna_vif,lesiones_casos_2018_2025,lesiones_tasa_prom_anual_100k,spa_abusivo_casos_2018_2025,spa_abusivo_tasa_prom_anual_100k,consumo_actual_spa_2022,indice_vif_familiar,indice_contexto,brecha_vif_contexto,rank_vif_carga,rank_vif_tasa,rank_vif_familiar,cambio_carga_vs_tasa,nivel_vif_familiar,nivel_contexto,tipo_decision,alerta_lectura,accion_sugerida,lectura_breve,lectura_breve_final
0,Los Mártires,LOS MARTIRES,99119,8819,1112.17,8,74.28,6496,819.22,7099,895.26,4.5,76.18,94.00,-17.82,12,1,1,11,alto,alto,VIF alta con entorno complejo,Media: posible efecto de población flotante o ...,Conviene revisar la oferta de atención familia...,"Registra 8.819 casos entre 2018 y 2025, con un...","Registra 8.819 casos entre 2018 y 2025, con un..."
1,Ciudad Bolívar,CIUDAD BOLIVAR,707569,32707,577.81,8,80.45,16160,285.48,4909,86.72,4.4,74.12,17.31,56.81,3,5,2,-2,alto,bajo,VIF alta con contexto moderado,Baja: lectura territorial más estable.,La prioridad debería estar en rutas de atenció...,"Registra 32.707 casos entre 2018 y 2025, con u...","Registra 32.707 casos entre 2018 y 2025, con u..."
2,La Candelaria,LA CANDELARIA,24088,2128,1104.28,8,72.46,1777,922.14,1634,847.93,8.2,70.28,97.33,-27.05,19,2,3,17,alto,alto,VIF alta con entorno complejo,Alta: población pequeña. Interpretar tasas con...,Conviene revisar la oferta de atención familia...,"Registra 2.128 casos entre 2018 y 2025, con un...","Registra 2.128 casos entre 2018 y 2025, con un..."
3,Bosa,BOSA,673077,31314,581.55,7,79.53,14132,262.45,5943,110.37,2.2,68.27,17.30,50.97,4,4,4,0,alto,bajo,VIF alta con contexto moderado,Baja: lectura territorial más estable.,La prioridad debería estar en rutas de atenció...,"Registra 31.314 casos entre 2018 y 2025, con u...","Registra 31.314 casos entre 2018 y 2025, con u..."
4,Santa Fe,SANTA FE,110048,7813,887.45,8,74.93,6441,731.61,4877,553.96,5.1,68.22,69.66,-1.44,13,3,5,10,alto,alto,VIF alta con entorno complejo,Media: posible efecto de población flotante o ...,Conviene revisar la oferta de atención familia...,"Registra 7.813 casos entre 2018 y 2025, con un...","Registra 7.813 casos entre 2018 y 2025, con un..."
5,San Cristóbal,SAN CRISTOBAL,404697,17723,547.42,8,79.32,9979,308.22,2993,92.45,4.1,66.31,18.96,47.35,6,6,6,0,alto,bajo,VIF alta con contexto moderado,Baja: lectura territorial más estable.,La prioridad debería estar en rutas de atenció...,"Registra 17.723 casos entre 2018 y 2025, con u...","Registra 17.723 casos entre 2018 y 2025, con u..."
6,Usme,USME,457302,15558,425.27,6,80.63,9623,263.04,3408,93.16,3.6,55.93,16.37,39.56,7,10,7,-3,medio,bajo,Seguimiento focalizado,Baja: lectura territorial más estable.,La VIF pesa más que las señales de contexto. V...,"Registra 15.558 casos entre 2018 y 2025, con u...","Registra 15.558 casos entre 2018 y 2025, con u..."
7,Kennedy,KENNEDY,1088443,34615,397.53,3,79.46,20577,236.31,7581,87.06,3.2,46.75,14.46,32.29,1,12,8,-11,medio,bajo,Seguimiento focalizado,Baja: lectura territorial más estable.,La VIF pesa más que las señales de contexto. V...,"Registra 34.615 casos entre 2018 y 2025, con u...","Registra 34.615 casos entre 2018 y 2025, con u..."
8,Tunjuelito,TUNJUELITO,199430,7002,438.88,4,77.71,4758,298.22,2472,154.94,3.3,40.66,21.90,18.76,14,9,9,5,medio,bajo,Seguimiento focalizado,Baja: lectura territorial más estable.,La VIF pesa más que las señales de contexto. V...,"Registra 7.002 casos entre 2018 y 2025, con un...","Registra 7.002 casos entre 2018 y 2025, con un..."
9,Rafael Uribe Uribe,RAFAEL URIBE URIBE,374246,12593,420.61,3,78.67,9394,313.76,2798,93.45,6.3,39.72,19.34,20.38,8,11,10,-3,medio,bajo,Seguimiento focalizado,Baja: lectura territorial más estable.,La VIF pesa más que las señales de contexto. V...,"Registra 12.593 casos entre 2018 y 2025, con u...","Registra 12.593 casos entre 2018 y 2025, con u..."


## 15. Validaciones de calidad

Estas validaciones son controles de entrega. Si alguna falla, conviene revisar la base limpia antes de publicar resultados.

In [15]:
validaciones = pd.DataFrame([
    {
        "validacion": "Localidades en base integrada",
        "valor_observado": int(base_integrada["loc_norm"].nunique()),
        "valor_esperado": 20,
        "estado": "OK" if base_integrada["loc_norm"].nunique() == 20 else "REVISAR",
    },
    {
        "validacion": "Casos VIF localizados",
        "valor_observado": int(base_integrada["vif_casos_2018_2025"].sum()),
        "valor_esperado": 283773,
        "estado": "OK" if int(base_integrada["vif_casos_2018_2025"].sum()) == 283773 else "REVISAR",
    },
    {
        "validacion": "Casos lesiones localizados",
        "valor_observado": int(base_integrada["lesiones_casos_2018_2025"].sum()),
        "valor_esperado": 162475,
        "estado": "OK" if int(base_integrada["lesiones_casos_2018_2025"].sum()) == 162475 else "REVISAR",
    },
    {
        "validacion": "Casos SPA abusivo localizados",
        "valor_observado": int(base_integrada["spa_abusivo_casos_2018_2025"].sum()),
        "valor_esperado": 74238,
        "estado": "OK" if int(base_integrada["spa_abusivo_casos_2018_2025"].sum()) == 74238 else "REVISAR",
    },
    {
        "validacion": "Persistencia máxima VIF",
        "valor_observado": int(base_integrada["vif_persistencia_anios"].max()),
        "valor_esperado": 8,
        "estado": "OK" if int(base_integrada["vif_persistencia_anios"].max()) == 8 else "REVISAR",
    },
    {
        "validacion": "Duplicados por localidad",
        "valor_observado": int(base_integrada["loc_norm"].duplicated().sum()),
        "valor_esperado": 0,
        "estado": "OK" if int(base_integrada["loc_norm"].duplicated().sum()) == 0 else "REVISAR",
    },
])

validaciones

,validacion,valor_observado,valor_esperado,estado
0,Localidades en base integrada,20,20,OK
1,Casos VIF localizados,283773,283773,OK
2,Casos lesiones localizados,162475,162475,OK
3,Casos SPA abusivo localizados,74238,74238,OK
4,Persistencia máxima VIF,8,8,OK
5,Duplicados por localidad,0,0,OK


## 16. Pruebas estadísticas exploratorias

Las pruebas revisan asociaciones territoriales entre indicadores. No se interpretan como causalidad.

In [16]:
def prueba_asociacion(df, x, y, nombre):
    datos = df[[x, y]].dropna()
    fila = {"comparacion": nombre, "x": x, "y": y, "n_localidades": len(datos)}

    if SCIPY_DISPONIBLE and len(datos) >= 3:
        rho, p_s = spearmanr(datos[x], datos[y])
        tau, p_k = kendalltau(datos[x], datos[y])
        r, p_p = pearsonr(datos[x], datos[y])
        fila.update({
            "spearman_rho": round(float(rho), 4),
            "spearman_p_value": round(float(p_s), 4),
            "kendall_tau": round(float(tau), 4),
            "kendall_p_value": round(float(p_k), 4),
            "pearson_r": round(float(r), 4),
            "pearson_p_value": round(float(p_p), 4),
        })
    else:
        fila.update({
            "spearman_rho": np.nan,
            "spearman_p_value": np.nan,
            "kendall_tau": np.nan,
            "kendall_p_value": np.nan,
            "pearson_r": np.nan,
            "pearson_p_value": np.nan,
        })

    return fila

resultados_pruebas = pd.DataFrame([
    prueba_asociacion(base_integrada, "indice_vif_familiar", "indice_contexto", "Índice VIF vs índice de contexto"),
    prueba_asociacion(base_integrada, "vif_tasa_prom_anual_100k", "lesiones_tasa_prom_anual_100k", "Tasa VIF vs tasa lesiones"),
    prueba_asociacion(base_integrada, "vif_tasa_prom_anual_100k", "spa_abusivo_tasa_prom_anual_100k", "Tasa VIF vs tasa SPA abusivo"),
])

resultados_pruebas

,comparacion,x,y,n_localidades,spearman_rho,spearman_p_value,kendall_tau,kendall_p_value,pearson_r,pearson_p_value
0,Índice VIF vs índice de contexto,indice_vif_familiar,indice_contexto,20,0.5489,0.0122,0.4000,0.0135,0.6155,0.0039
1,Tasa VIF vs tasa lesiones,vif_tasa_prom_anual_100k,lesiones_tasa_prom_anual_100k,20,0.7970,0.0000,0.6105,0.0001,0.9521,0.0000
2,Tasa VIF vs tasa SPA abusivo,vif_tasa_prom_anual_100k,spa_abusivo_tasa_prom_anual_100k,20,0.7338,0.0002,0.5895,0.0001,0.9305,0.0000


## 17. Revisiónr HTML

Si el archivo HTML está disponible, se extrae el bloque `DATA` para comparar los indicadores comunes. El cuaderno solo lee el HTML; no lo modifica.

In [17]:
def extraer_data_html(html_path):
    texto = Path(html_path).read_text(encoding="utf-8")
    patron = re.search(r"const DATA = (\[.*?\]);\s*\n\s*const fmt", texto, flags=re.S)
    if not patron:
        raise ValueError("No se encontró el bloque const DATA en el HTML.")
    return pd.DataFrame(json.loads(patron.group(1)))

comparacion_html = pd.DataFrame()

if HTML_PATH is not None:
    data_html = extraer_data_html(HTML_PATH)
    comunes = [c for c in base_integrada.columns if c in data_html.columns]
    merged_html = base_integrada[comunes].merge(
        data_html[comunes],
        on="loc_norm",
        suffixes=("_calculado", "_html")
    )

    filas = []
    columnas_numericas = [
        "vif_casos_2018_2025", "vif_tasa_prom_anual_100k", "vif_persistencia_anios", "pct_mujeres_nna_vif",
        "lesiones_casos_2018_2025", "lesiones_tasa_prom_anual_100k",
        "spa_abusivo_casos_2018_2025", "spa_abusivo_tasa_prom_anual_100k",
        "indice_vif_familiar", "indice_contexto", "brecha_vif_contexto",
        "rank_vif_carga", "rank_vif_tasa", "rank_vif_familiar", "cambio_carga_vs_tasa",
    ]

    for col in columnas_numericas:
        if f"{col}_calculado" in merged_html.columns and f"{col}_html" in merged_html.columns:
            dif = (
                pd.to_numeric(merged_html[f"{col}_calculado"], errors="coerce") -
                pd.to_numeric(merged_html[f"{col}_html"], errors="coerce")
            ).abs()
            filas.append({
                "campo": col,
                "diferencia_maxima": round(float(dif.max()), 4),
                "estado": "OK" if float(dif.max()) <= 0.05 else "REVISAR",
            })

    for col in ["nivel_vif_familiar", "nivel_contexto", "tipo_decision"]:
        if f"{col}_calculado" in merged_html.columns and f"{col}_html" in merged_html.columns:
            distintos = int((merged_html[f"{col}_calculado"] != merged_html[f"{col}_html"]).sum())
            filas.append({
                "campo": col,
                "diferencia_maxima": distintos,
                "estado": "OK" if distintos == 0 else "REVISAR",
            })

    comparacion_html = pd.DataFrame(filas)
    print("HTML revisado:", HTML_PATH.name)
    display(comparacion_html)
else:
    print("No se encontró HTML. Se omite la comparación con el visor.")

No se encontró HTML. Se omite la comparación con el visor.


## 18. Exportación de archivos

Se exportan los archivos que alimentan la entrega técnica. El HTML no se edita desde este cuaderno.

In [18]:
base_integrada.to_csv(OUTPUT_DIR / "base_integrada_localidad_2018_2025.csv", index=False, encoding="utf-8-sig")
validaciones.to_csv(OUTPUT_DIR / "validaciones_calidad.csv", index=False, encoding="utf-8-sig")
resultados_pruebas.to_csv(OUTPUT_DIR / "resultados_pruebas_estadisticas.csv", index=False, encoding="utf-8-sig")
poblacion_referencia.to_csv(OUTPUT_DIR / "poblacion_referencia_localidades.csv", index=False, encoding="utf-8-sig")

if comparacion_html is not None and not comparacion_html.empty:
    comparacion_html.to_csv(OUTPUT_DIR / "comparacion_base_vs_html.csv", index=False, encoding="utf-8-sig")

fuentes_resumen = pd.DataFrame([
    {"fuente": "Violencia intrafamiliar", "casos_localizados": int(base_integrada["vif_casos_2018_2025"].sum()), "uso": "Fenómeno principal"},
    {"fuente": "Lesiones personales", "casos_localizados": int(base_integrada["lesiones_casos_2018_2025"].sum()), "uso": "Señal de contexto"},
    {"fuente": "Consumo abusivo SPA", "casos_localizados": int(base_integrada["spa_abusivo_casos_2018_2025"].sum()), "uso": "Señal de contexto"},
    {"fuente": "Prevalencia consumo SPA", "casos_localizados": np.nan, "uso": "Insumo exploratorio 2022"},
    {"fuente": "RNMC", "casos_localizados": np.nan, "uso": "Revisada, no usada en el visor actual"},
])
fuentes_resumen.to_csv(OUTPUT_DIR / "fuentes_resumen.csv", index=False, encoding="utf-8-sig")

print("Archivos exportados en:", OUTPUT_DIR)
for archivo in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", archivo.name)

Archivos exportados en: /content/outputs
- base_integrada_localidad_2018_2025.csv
- fuentes_resumen.csv
- poblacion_referencia_localidades.csv
- resultados_pruebas_estadisticas.csv
- validaciones_calidad.csv


## 19. Resumen final

La base integrada queda lista

In [19]:
print("Resumen de salida")
print("Localidades:", base_integrada["loc_norm"].nunique())
print("Total VIF:", int(base_integrada["vif_casos_2018_2025"].sum()))
print("Total lesiones:", int(base_integrada["lesiones_casos_2018_2025"].sum()))
print("Total SPA abusivo:", int(base_integrada["spa_abusivo_casos_2018_2025"].sum()))
print("Mayor índice VIF:", base_integrada.sort_values("indice_vif_familiar", ascending=False).iloc[0]["localidad"])
print("Mayor brecha VIF-contexto:", base_integrada.sort_values("brecha_vif_contexto", ascending=False).iloc[0]["localidad"])

base_integrada[[
    "localidad", "vif_casos_2018_2025", "vif_tasa_prom_anual_100k",
    "indice_vif_familiar", "indice_contexto", "brecha_vif_contexto",
    "tipo_decision"
]].head(10)

Resumen de salida
Localidades: 20
Total VIF: 283773
Total lesiones: 162475
Total SPA abusivo: 74238
Mayor índice VIF: Los Mártires
Mayor brecha VIF-contexto: Ciudad Bolívar


,localidad,vif_casos_2018_2025,vif_tasa_prom_anual_100k,indice_vif_familiar,indice_contexto,brecha_vif_contexto,tipo_decision
0,Los Mártires,8819,1112.17,76.18,94.00,-17.82,VIF alta con entorno complejo
1,Ciudad Bolívar,32707,577.81,74.12,17.31,56.81,VIF alta con contexto moderado
2,La Candelaria,2128,1104.28,70.28,97.33,-27.05,VIF alta con entorno complejo
3,Bosa,31314,581.55,68.27,17.30,50.97,VIF alta con contexto moderado
4,Santa Fe,7813,887.45,68.22,69.66,-1.44,VIF alta con entorno complejo
5,San Cristóbal,17723,547.42,66.31,18.96,47.35,VIF alta con contexto moderado
6,Usme,15558,425.27,55.93,16.37,39.56,Seguimiento focalizado
7,Kennedy,34615,397.53,46.75,14.46,32.29,Seguimiento focalizado
8,Tunjuelito,7002,438.88,40.66,21.90,18.76,Seguimiento focalizado
9,Rafael Uribe Uribe,12593,420.61,39.72,19.34,20.38,Seguimiento focalizado
